# MOOCCubeX — EDA phase activity, lifecycle và eligibility

Notebook CPU-only này chạy sau `01_moocubex_raw_to_parquet.ipynb` và
`02_build_dimensions_bridges.ipynb`.

Mục tiêu:

- kiểm tra observation range của video, problem, comment và reply;
- chia log vào 5 fixed calendar phases theo khoảng half-open;
- tạo activity facts sơ bộ ở ba grain user–course, course–chapter và course;
- phân biệt learning activity, social activity, inactivity và source censoring;
- đo lifecycle, chapter jump/revisit và candidate eligibility;
- ước lượng kích thước ma trận feature trước khi tính COELO/ACELO/AFELO-proxy/CQV.

Notebook **không tạo nhãn chất lượng cuối**. Những chapter không có event được
giữ là inactive/unknown, không tự động gán `CQV=RED`.



In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

try:
    import duckdb
except ModuleNotFoundError:
    print("DuckDB chưa có; đang cài duckdb>=1.3 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.3"])
    import duckdb

# Chỉ đặt path khi auto-discovery tìm thấy nhiều dataset phù hợp.
RAW_PARQUET_ROOT: Optional[str] = None
MODEL_ROOT: Optional[str] = None

OUTPUT_ROOT = Path("/kaggle/working/moocubex_eda")
FORCE_REBUILD = False

# Giữ thấp hơn notebook 02 vì UNNEST + GROUP BY user có cardinality rất lớn.
# Low-memory defaults cho Kaggle CPU Free. Chạy lâu hơn nhưng giảm peak RAM.
THREADS = 1
MEMORY_FRACTION = 0.20
USER_HASH_SHARDS = 256
ROW_GROUP_SIZE = 50_000
ZSTD_LEVEL = 3
VIDEO_COVERAGE_SAMPLE_PERCENT = 1

# Chỉ dùng làm candidate EDA, chưa phải threshold nhãn cuối.
MIN_ACTIVE_USERS_CANDIDATES = [1, 5, 10, 20]

STUDY_START = "2020-01-19 00:00:00"
STUDY_END_EXCLUSIVE = "2020-11-04 00:00:00"

PHASES = [
    ("P1", 1, "2020-01-19 00:00:00", "2020-03-17 00:00:00"),
    ("P2", 2, "2020-03-17 00:00:00", "2020-05-14 00:00:00"),
    ("P3", 3, "2020-05-14 00:00:00", "2020-07-11 00:00:00"),
    ("P4", 4, "2020-07-11 00:00:00", "2020-09-07 00:00:00"),
    ("P5", 5, "2020-09-07 00:00:00", "2020-11-04 00:00:00"),
]

print("Python:", sys.version.split()[0])
print("DuckDB:", duckdb.__version__)
print("THREADS:", THREADS)
print("USER_HASH_SHARDS:", USER_HASH_SHARDS)



## 1. Tự tìm output của notebook 01 và 02

Không dùng absolute path lưu trong catalog cũ, vì khi publish thành Kaggle Dataset
đường dẫn `/kaggle/working/...` sẽ đổi thành `/kaggle/input/...`.



In [ ]:
def is_raw_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "catalog.json").is_file()
        and (path / "tables").is_dir()
        and (path / "tables" / "relations" / "user_video").is_dir()
    )


def is_model_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "catalog.json").is_file()
        and (path / "processed" / "dim_lo_catalog").is_dir()
    )


def discover_root(configured: Optional[str], kind: str) -> Path:
    predicate = is_raw_root if kind == "raw" else is_model_root
    if configured:
        root = Path(configured).resolve()
        if not predicate(root):
            raise FileNotFoundError(f"{kind} root không đúng layout: {root}")
        return root

    candidates: List[Path] = []
    direct_paths = (
        [Path("/kaggle/working/moocubex_parquet")]
        if kind == "raw"
        else [Path("/kaggle/working/moocubex_model")]
    )
    for path in direct_paths:
        if predicate(path):
            candidates.append(path.resolve())

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for catalog_path in kaggle_input.rglob("catalog.json"):
            root = catalog_path.parent.resolve()
            if predicate(root) and root not in candidates:
                candidates.append(root)

    if not candidates:
        expected = "moocubex_parquet" if kind == "raw" else "moocubex_model"
        raise FileNotFoundError(
            f"Không tìm thấy {kind} root. Hãy attach Kaggle Dataset chứa output "
            f"{expected}, hoặc đặt path cấu hình tương ứng."
        )

    print(f"Các {kind} root tìm thấy:")
    for index, candidate in enumerate(candidates):
        print(f" [{index}] {candidate}")
    if len(candidates) > 1:
        print(f"Dùng candidate đầu tiên cho {kind}; đặt path nếu muốn chọn root khác.")
    return candidates[0]


RAW_ROOT_PATH = discover_root(RAW_PARQUET_ROOT, "raw")
MODEL_ROOT_PATH = discover_root(MODEL_ROOT, "model")


def raw_table_dir(logical_path: str) -> Path:
    rel = Path(logical_path)
    clean_parts = [part.replace("-", "_") for part in rel.with_suffix("").parts]
    path = RAW_ROOT_PATH / "tables"
    for part in clean_parts:
        path = path / part
    return path


def raw_table_exists(logical_path: str) -> bool:
    return any(raw_table_dir(logical_path).glob("*.parquet"))


def raw_table_glob(logical_path: str) -> str:
    path = raw_table_dir(logical_path)
    if not any(path.glob("*.parquet")):
        raise FileNotFoundError(f"Thiếu raw table {logical_path}: {path}")
    return str(path / "*.parquet")


def model_table_dir(table_name: str) -> Path:
    return MODEL_ROOT_PATH / "processed" / table_name


def model_table_exists(table_name: str) -> bool:
    return any(model_table_dir(table_name).glob("*.parquet"))


def model_table_glob(table_name: str) -> str:
    path = model_table_dir(table_name)
    if not any(path.glob("*.parquet")):
        raise FileNotFoundError(f"Thiếu model table {table_name}: {path}")
    return str(path / "*.parquet")


REQUIRED_RAW = [
    "relations/user-video.json",
    "relations/user-problem.json",
]
REQUIRED_MODEL = [
    "dim_course",
    "dim_lo_catalog",
    "dim_lo_resource",
    "bridge_lo_problem",
    "bridge_user_course_enrollment",
]

missing_raw = [name for name in REQUIRED_RAW if not raw_table_exists(name)]
missing_model = [name for name in REQUIRED_MODEL if not model_table_exists(name)]
if missing_raw or missing_model:
    raise FileNotFoundError(
        "Thiếu input bắt buộc. "
        f"raw={missing_raw or 'OK'}, model={missing_model or 'OK'}"
    )

print("RAW_ROOT   =", RAW_ROOT_PATH)
print("MODEL_ROOT =", MODEL_ROOT_PATH)



## 2. DuckDB out-of-core, source views và checkpoint writer



In [ ]:
def total_memory_bytes() -> int:
    candidates: List[int] = []
    try:
        candidates.append(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES"))
    except (ValueError, OSError, AttributeError):
        pass

    # Kaggle chạy trong container; host RAM có thể lớn hơn cgroup limit thật.
    for limit_path in [
        Path("/sys/fs/cgroup/memory.max"),
        Path("/sys/fs/cgroup/memory/memory.limit_in_bytes"),
    ]:
        try:
            raw = limit_path.read_text(encoding="utf-8").strip()
            if raw != "max":
                value = int(raw)
                if 0 < value < 2**60:
                    candidates.append(value)
        except (OSError, ValueError):
            pass

    return min(candidates) if candidates else 30 * 1024**3


total_mem = total_memory_bytes()
memory_limit_gib = max(2, int(total_mem * MEMORY_FRACTION / 1024**3))
scratch_base = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
DUCKDB_TEMP = scratch_base / "moocubex_eda_duckdb"
DUCKDB_TEMP.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "figures").mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET memory_limit='{memory_limit_gib}GB'")
con.execute(f"SET temp_directory='{str(DUCKDB_TEMP).replace(chr(39), chr(39) * 2)}'")
con.execute("SET preserve_insertion_order=false")
con.execute("SET partitioned_write_max_open_files=16")
con.execute("SET enable_progress_bar=true")
con.execute("SET TimeZone='UTC'")

print(f"DuckDB memory_limit = {memory_limit_gib} GiB")
print("DuckDB temp_directory =", DUCKDB_TEMP)
disk_path = Path("/kaggle/working") if Path("/kaggle/working").exists() else OUTPUT_ROOT.parent
print("Free output filesystem =", f"{shutil.disk_usage(disk_path).free / 1024**3:.2f} GiB")


def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def normalize_id(expr: str, prefix: str) -> str:
    return f"""
        CASE
          WHEN {expr} IS NULL OR trim(CAST({expr} AS VARCHAR)) = '' THEN NULL
          WHEN starts_with(trim(CAST({expr} AS VARCHAR)), '{prefix}')
            THEN trim(CAST({expr} AS VARCHAR))
          ELSE '{prefix}' || trim(CAST({expr} AS VARCHAR))
        END
    """


RAW_VIEWS = {
    "src_user_video": "relations/user-video.json",
    "src_user_problem": "relations/user-problem.json",
}
OPTIONAL_RAW_VIEWS = {
    "src_comment": "entities/comment.json",
    "src_reply": "entities/reply.json",
}
MODEL_VIEWS = {name: name for name in REQUIRED_MODEL}
OPTIONAL_MODEL_VIEWS = {
    "bridge_comment_resource": "bridge_comment_resource",
    "bridge_course_comment": "bridge_course_comment",
    "bridge_user_comment": "bridge_user_comment",
}

for view_name, logical_path in RAW_VIEWS.items():
    con.execute(
        f"CREATE OR REPLACE VIEW {view_name} AS "
        f"SELECT * FROM read_parquet({sql_string(raw_table_glob(logical_path))}, union_by_name=true)"
    )

AVAILABLE_RAW: Dict[str, bool] = {}
for view_name, logical_path in OPTIONAL_RAW_VIEWS.items():
    available = raw_table_exists(logical_path)
    AVAILABLE_RAW[view_name] = available
    if available:
        con.execute(
            f"CREATE OR REPLACE VIEW {view_name} AS "
            f"SELECT * FROM read_parquet({sql_string(raw_table_glob(logical_path))}, union_by_name=true)"
        )
    print((" OK " if available else " -- "), logical_path)

for view_name, table_name in MODEL_VIEWS.items():
    con.execute(
        f"CREATE OR REPLACE VIEW {view_name} AS "
        f"SELECT * FROM read_parquet({sql_string(model_table_glob(table_name))}, union_by_name=true)"
    )

AVAILABLE_MODEL: Dict[str, bool] = {}
for view_name, table_name in OPTIONAL_MODEL_VIEWS.items():
    available = model_table_exists(table_name)
    AVAILABLE_MODEL[view_name] = available
    if available:
        con.execute(
            f"CREATE OR REPLACE VIEW {view_name} AS "
            f"SELECT * FROM read_parquet({sql_string(model_table_glob(table_name))}, union_by_name=true)"
        )
    print((" OK " if available else " -- "), table_name)

required_columns = {
    "src_user_video": {"user_id", "seq"},
    "src_user_problem": {
        "problem_id", "user_id", "is_correct", "attempts", "score", "submit_time"
    },
    "dim_lo_catalog": {"course_id", "chapter_major", "lo_key"},
    "dim_lo_resource": {"course_id", "chapter_major", "lo_key", "resource_id", "resource_type"},
    "bridge_lo_problem": {"course_id", "chapter_major", "lo_key", "problem_id"},
}
for view_name, expected in required_columns.items():
    actual = {row[0] for row in con.execute(f"DESCRIBE {view_name}").fetchall()}
    missing = expected - actual
    if missing:
        raise ValueError(f"{view_name} thiếu cột {sorted(missing)}; hiện có {sorted(actual)}")
    print("SCHEMA OK", view_name)

phase_values = ",\n".join(
    f"({sql_string(pid)}, {pnum}, TIMESTAMP {sql_string(start)}, TIMESTAMP {sql_string(end)})"
    for pid, pnum, start, end in PHASES
)
con.execute(f"""
CREATE OR REPLACE TEMP VIEW phase_dim AS
SELECT * FROM (VALUES {phase_values}) AS p(phase_id, phase_num, phase_start, phase_end)
""")


def time_bucket_sql(event_time_expr: str) -> str:
    cases = [
        f"WHEN {event_time_expr} >= TIMESTAMP {sql_string(start)} "
        f"AND {event_time_expr} < TIMESTAMP {sql_string(end)} THEN {sql_string(pid)}"
        for pid, _, start, end in PHASES
    ]
    return f"""
        CASE
          WHEN {event_time_expr} IS NULL THEN 'INVALID_TIME'
          WHEN {event_time_expr} < TIMESTAMP {sql_string(STUDY_START)} THEN 'BEFORE_T0'
          {' '.join(cases)}
          WHEN {event_time_expr} >= TIMESTAMP {sql_string(STUDY_END_EXCLUSIVE)} THEN 'AFTER_T5'
          ELSE 'UNASSIGNED'
        END
    """


MANIFEST_PATH = OUTPUT_ROOT / "build_manifest.json"


def load_manifest() -> Dict[str, dict]:
    if not MANIFEST_PATH.exists():
        return {}
    with MANIFEST_PATH.open("r", encoding="utf-8") as file:
        rows = json.load(file)
    return {row["name"]: row for row in rows}


build_manifest = load_manifest()


def save_manifest() -> None:
    rows = sorted(build_manifest.values(), key=lambda row: row["name"])
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as file:
        json.dump(rows, file, ensure_ascii=False, indent=2)
    tmp.replace(MANIFEST_PATH)


def register_output(view_name: str, out_dir: Path) -> None:
    con.execute(
        f"CREATE OR REPLACE VIEW {view_name} AS "
        f"SELECT * FROM read_parquet({sql_string(str(out_dir / '*.parquet'))}, union_by_name=true)"
    )


def remove_scoped_output(out_dir: Path) -> None:
    root = OUTPUT_ROOT.resolve()
    target = out_dir.resolve()
    if root not in target.parents:
        raise RuntimeError(f"Từ chối xóa path ngoài OUTPUT_ROOT: {target}")
    if out_dir.exists():
        shutil.rmtree(out_dir)


def output_has_columns(out_dir: Path, required_columns) -> bool:
    if not required_columns or not any(out_dir.glob("*.parquet")):
        return not required_columns
    try:
        actual = {
            row[0]
            for row in con.execute(
                f"DESCRIBE SELECT * FROM read_parquet("
                f"{sql_string(str(out_dir / '*.parquet'))}, union_by_name=true)"
            ).fetchall()
        }
        return set(required_columns).issubset(actual)
    except Exception:
        return False


def write_relation(
    name: str,
    relative_path: str,
    query: str,
    required_columns=None,
) -> dict:
    out_dir = OUTPUT_ROOT / relative_path
    out_file = out_dir / "part-00000.parquet"
    previous = build_manifest.get(name)

    if (
        not FORCE_REBUILD
        and previous
        and previous.get("status") == "success"
        and any(out_dir.glob("*.parquet"))
        and output_has_columns(out_dir, required_columns)
    ):
        register_output(name, out_dir)
        print(f"SKIP {name}: rows={previous.get('row_count', 0):,}")
        return previous

    remove_scoped_output(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    record = {
        "name": name,
        "relative_path": relative_path,
        "status": "running",
        "row_count": None,
        "output_bytes": 0,
        "elapsed_seconds": None,
        "error": None,
    }
    build_manifest[name] = record
    save_manifest()

    print("BUILD", name)
    try:
        con.execute(
            f"""
            COPY ({query})
            TO {sql_string(str(out_file))}
            (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL {ZSTD_LEVEL},
             ROW_GROUP_SIZE {ROW_GROUP_SIZE})
            """
        )
        register_output(name, out_dir)
        row_count = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
        output_bytes = sum(path.stat().st_size for path in out_dir.glob("*.parquet"))
        record.update({
            "status": "success",
            "row_count": int(row_count),
            "output_bytes": int(output_bytes),
            "elapsed_seconds": round(time.time() - started, 3),
        })
        print(
            f" OK rows={row_count:,} size={output_bytes / 1024**2:.1f} MiB "
            f"time={record['elapsed_seconds']:.1f}s"
        )
    except Exception as exc:
        record.update({
            "status": "error",
            "elapsed_seconds": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
        })
        print("ERROR", name, record["error"])
        raise
    finally:
        build_manifest[name] = record
        save_manifest()
    return record


def write_hash_sharded_relation(
    name: str,
    relative_path: str,
    query_builder,
    shard_count: int = USER_HASH_SHARDS,
) -> dict:
    """Ghi từng user-hash shard và resume an toàn sau khi runtime bị ngắt."""
    out_dir = OUTPUT_ROOT / relative_path
    previous = build_manifest.get(name, {})
    expected_files = [out_dir / f"part-shard-{i:05d}.parquet" for i in range(shard_count)]

    if (
        not FORCE_REBUILD
        and previous.get("status") == "success"
        and all(path.is_file() for path in expected_files)
    ):
        register_output(name, out_dir)
        print(f"SKIP {name}: rows={previous.get('row_count', 0):,}, shards={shard_count}")
        return previous

    shard_layout_changed = (
        previous.get("shard_count") is not None
        and int(previous["shard_count"]) != shard_count
    )
    if FORCE_REBUILD or shard_layout_changed:
        remove_scoped_output(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Xóa file đơn của phiên bản trước; chỉ các part-shard hoàn tất mới được resume.
    for legacy in out_dir.glob("*.parquet"):
        if not legacy.name.startswith("part-shard-"):
            legacy.unlink()

    started = time.time()
    record = {
        "name": name,
        "relative_path": relative_path,
        "status": "running",
        "row_count": None,
        "output_bytes": 0,
        "elapsed_seconds": None,
        "completed_shards": [],
        "shard_count": shard_count,
        "error": None,
    }
    build_manifest[name] = record
    save_manifest()

    print(f"BUILD SHARDED {name}: {shard_count} user shards")
    try:
        for shard_id, final_file in enumerate(expected_files):
            if final_file.is_file() and not FORCE_REBUILD:
                record["completed_shards"].append(shard_id)
                print(f" SKIP shard {shard_id + 1}/{shard_count}")
                continue

            temp_file = out_dir / f".part-shard-{shard_id:05d}.tmp"
            if temp_file.exists():
                temp_file.unlink()

            shard_started = time.time()
            shard_query = query_builder(shard_id, shard_count)
            con.execute(
                f"""
                COPY ({shard_query})
                TO {sql_string(str(temp_file))}
                (FORMAT PARQUET, COMPRESSION ZSTD, COMPRESSION_LEVEL {ZSTD_LEVEL},
                 ROW_GROUP_SIZE {ROW_GROUP_SIZE})
                """
            )
            temp_file.replace(final_file)
            record["completed_shards"].append(shard_id)
            record["output_bytes"] = sum(path.stat().st_size for path in expected_files if path.exists())
            record["elapsed_seconds"] = round(time.time() - started, 3)
            build_manifest[name] = record
            save_manifest()
            print(
                f" OK shard {shard_id + 1}/{shard_count} "
                f"size={final_file.stat().st_size / 1024**2:.1f} MiB "
                f"time={time.time() - shard_started:.1f}s"
            )

        register_output(name, out_dir)
        row_count = con.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
        record.update({
            "status": "success",
            "row_count": int(row_count),
            "output_bytes": int(sum(path.stat().st_size for path in expected_files)),
            "elapsed_seconds": round(time.time() - started, 3),
            "completed_shards": list(range(shard_count)),
            "error": None,
        })
        print(
            f" OK {name}: rows={row_count:,}, shards={shard_count}, "
            f"size={record['output_bytes'] / 1024**2:.1f} MiB"
        )
    except Exception as exc:
        record.update({
            "status": "error",
            "elapsed_seconds": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
        })
        print("ERROR", name, record["error"])
        raise
    finally:
        build_manifest[name] = record
        save_manifest()
    return record



## 3. Mapping strict tới big chapter

EDA activity cấp chapter chỉ dùng resource/problem có đúng một assignment
`course × lo`. Mapping mơ hồ vẫn được đếm trong báo cáo coverage, nhưng không bị
nhân bản sang nhiều chapter.



In [ ]:
con.execute("""
CREATE OR REPLACE TEMP TABLE map_video_lo AS
SELECT
    resource_id AS video_id,
    min(course_id) AS course_id_candidate,
    min(lo_key) AS lo_key_candidate,
    min(chapter_major) AS chapter_major_candidate,
    count(DISTINCT course_id) AS n_course_matches,
    count(DISTINCT lo_key) AS n_lo_matches
FROM dim_lo_resource
WHERE resource_type = 'video'
GROUP BY resource_id
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE map_problem_lo AS
SELECT
    problem_id,
    min(course_id) AS course_id_candidate,
    min(lo_key) AS lo_key_candidate,
    min(chapter_major) AS chapter_major_candidate,
    count(DISTINCT course_id) AS n_course_matches,
    count(DISTINCT lo_key) AS n_lo_matches
FROM bridge_lo_problem
GROUP BY problem_id
""")

if AVAILABLE_MODEL.get("bridge_user_comment"):
    con.execute("""
    CREATE OR REPLACE TEMP TABLE map_comment_user AS
    SELECT
        comment_id,
        min(user_id) AS user_id_candidate,
        count(DISTINCT user_id) AS n_user_matches
    FROM bridge_user_comment
    GROUP BY comment_id
    """)
else:
    con.execute("""
    CREATE OR REPLACE TEMP TABLE map_comment_user AS
    SELECT CAST(NULL AS VARCHAR) AS comment_id,
           CAST(NULL AS VARCHAR) AS user_id_candidate,
           CAST(0 AS BIGINT) AS n_user_matches
    WHERE FALSE
    """)

comment_course_candidates: List[str] = []
if AVAILABLE_MODEL.get("bridge_course_comment"):
    comment_course_candidates.append(
        "SELECT comment_id, course_id FROM bridge_course_comment WHERE course_id IS NOT NULL"
    )
if AVAILABLE_MODEL.get("bridge_comment_resource"):
    comment_course_candidates.append(
        "SELECT comment_id, resolved_course_id AS course_id FROM bridge_comment_resource "
        "WHERE resolved_course_id IS NOT NULL"
    )

if comment_course_candidates:
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE map_comment_course AS
    WITH candidates AS ({' UNION ALL '.join(comment_course_candidates)})
    SELECT
        comment_id,
        min(course_id) AS course_id_candidate,
        count(DISTINCT course_id) AS n_course_matches
    FROM candidates
    GROUP BY comment_id
    """)
else:
    con.execute("""
    CREATE OR REPLACE TEMP TABLE map_comment_course AS
    SELECT CAST(NULL AS VARCHAR) AS comment_id,
           CAST(NULL AS VARCHAR) AS course_id_candidate,
           CAST(0 AS BIGINT) AS n_course_matches
    WHERE FALSE
    """)



## 4. Audit thời gian và chất lượng từng source

`time_bucket` ngoài P1–P5 giúp nhận diện source không cùng observation window.
Video dùng epoch seconds và được diễn giải ở UTC; giả định này phải được giữ như
một data-quality assumption cho tới khi có tài liệu timezone chắc chắn hơn.



In [ ]:
video_user_id = normalize_id("uv.user_id", "U_")
video_id = normalize_id("video_item.video_id", "V_")
video_time_bucket = time_bucket_sql("event_time")
write_relation(
    "eda_video_source_phase_quality",
    "source/eda_video_source_phase_quality",
    f"""
    WITH flat AS (
      SELECT
          {video_user_id} AS user_id,
          {video_id} AS video_id,
          segment_item.start_point AS start_point,
          segment_item.end_point AS end_point,
          segment_item.speed AS speed,
          segment_item.local_start_time AS local_start_time,
          CASE
            WHEN segment_item.local_start_time BETWEEN 0 AND 4102444800
              THEN CAST(to_timestamp(segment_item.local_start_time) AS TIMESTAMP)
            ELSE NULL
          END AS event_time
      FROM src_user_video uv
      CROSS JOIN UNNEST(uv.seq) AS s(video_item)
      CROSS JOIN UNNEST(video_item.segment) AS g(segment_item)
    ), bucketed AS (
      SELECT *, {video_time_bucket} AS time_bucket
      FROM flat
    )
    SELECT
        'video' AS source_name,
        time_bucket,
        count(*) AS event_count,
        approx_count_distinct(user_id) AS n_users,
        min(event_time) AS min_event_time,
        max(event_time) AS max_event_time,
        count(*) FILTER (WHERE event_time IS NULL) AS invalid_time_count,
        count(*) FILTER (
          WHERE start_point IS NULL OR end_point IS NULL OR end_point <= start_point
        ) AS invalid_interval_count,
        count(*) FILTER (WHERE speed IS NULL OR speed <= 0) AS invalid_speed_count,
        TRUE AS source_available
    FROM bucketed
    GROUP BY time_bucket
    """,
)

problem_user_id = normalize_id("up.user_id", "U_")
problem_id = normalize_id("up.problem_id", "Pm_")
problem_time_bucket = time_bucket_sql("event_time")
write_relation(
    "eda_problem_source_phase_quality",
    "source/eda_problem_source_phase_quality",
    f"""
    WITH typed AS (
      SELECT
          {problem_user_id} AS user_id,
          {problem_id} AS problem_id,
          try_cast(up.submit_time AS TIMESTAMP) AS event_time
      FROM src_user_problem up
    ), bucketed AS (
      SELECT *, {problem_time_bucket} AS time_bucket
      FROM typed
    )
    SELECT
        'problem' AS source_name,
        time_bucket,
        count(*) AS event_count,
        approx_count_distinct(user_id) AS n_users,
        min(event_time) AS min_event_time,
        max(event_time) AS max_event_time,
        count(*) FILTER (WHERE event_time IS NULL) AS invalid_time_count,
        CAST(0 AS BIGINT) AS invalid_interval_count,
        CAST(0 AS BIGINT) AS invalid_speed_count,
        TRUE AS source_available
    FROM bucketed
    GROUP BY time_bucket
    """,
)

comment_id_expr = normalize_id("cm.id", "Cm_")
comment_user_id_expr = normalize_id("cm.user_id", "U_")
comment_time_bucket = time_bucket_sql("event_time")
if AVAILABLE_RAW.get("src_comment"):
    comment_source_query = f"""
    WITH typed AS (
      SELECT
          {comment_id_expr} AS comment_id,
          {comment_user_id_expr} AS user_id,
          try_cast(cm.create_time AS TIMESTAMP) AS event_time
      FROM src_comment cm
    ), bucketed AS (
      SELECT *, {comment_time_bucket} AS time_bucket
      FROM typed
    )
    SELECT
        'comment' AS source_name,
        time_bucket,
        count(*) AS event_count,
        approx_count_distinct(user_id) AS n_users,
        min(event_time) AS min_event_time,
        max(event_time) AS max_event_time,
        count(*) FILTER (WHERE event_time IS NULL) AS invalid_time_count,
        CAST(0 AS BIGINT) AS invalid_interval_count,
        CAST(0 AS BIGINT) AS invalid_speed_count,
        TRUE AS source_available
    FROM bucketed
    GROUP BY time_bucket
    """
else:
    comment_source_query = """
    SELECT 'comment' AS source_name, 'SOURCE_MISSING' AS time_bucket,
           CAST(0 AS BIGINT) AS event_count, CAST(0 AS BIGINT) AS n_users,
           CAST(NULL AS TIMESTAMP) AS min_event_time,
           CAST(NULL AS TIMESTAMP) AS max_event_time,
           CAST(0 AS BIGINT) AS invalid_time_count,
           CAST(0 AS BIGINT) AS invalid_interval_count,
           CAST(0 AS BIGINT) AS invalid_speed_count,
           FALSE AS source_available
    """
write_relation(
    "eda_comment_source_phase_quality",
    "source/eda_comment_source_phase_quality",
    comment_source_query,
)

reply_id_expr = normalize_id("rp.id", "Rp_")
reply_user_id_expr = normalize_id("rp.user_id", "U_")
reply_time_bucket = time_bucket_sql("event_time")
if AVAILABLE_RAW.get("src_reply"):
    reply_source_query = f"""
    WITH typed AS (
      SELECT
          {reply_id_expr} AS reply_id,
          {reply_user_id_expr} AS user_id,
          try_cast(rp.create_time AS TIMESTAMP) AS event_time
      FROM src_reply rp
    ), bucketed AS (
      SELECT *, {reply_time_bucket} AS time_bucket
      FROM typed
    )
    SELECT
        'reply' AS source_name,
        time_bucket,
        count(*) AS event_count,
        approx_count_distinct(user_id) AS n_users,
        min(event_time) AS min_event_time,
        max(event_time) AS max_event_time,
        count(*) FILTER (WHERE event_time IS NULL) AS invalid_time_count,
        CAST(0 AS BIGINT) AS invalid_interval_count,
        CAST(0 AS BIGINT) AS invalid_speed_count,
        TRUE AS source_available
    FROM bucketed
    GROUP BY time_bucket
    """
else:
    reply_source_query = """
    SELECT 'reply' AS source_name, 'SOURCE_MISSING' AS time_bucket,
           CAST(0 AS BIGINT) AS event_count, CAST(0 AS BIGINT) AS n_users,
           CAST(NULL AS TIMESTAMP) AS min_event_time,
           CAST(NULL AS TIMESTAMP) AS max_event_time,
           CAST(0 AS BIGINT) AS invalid_time_count,
           CAST(0 AS BIGINT) AS invalid_interval_count,
           CAST(0 AS BIGINT) AS invalid_speed_count,
           FALSE AS source_available
    """
write_relation(
    "eda_reply_source_phase_quality",
    "source/eda_reply_source_phase_quality",
    reply_source_query,
)

write_relation(
    "eda_source_phase_coverage",
    "source/eda_source_phase_coverage",
    """
    SELECT * FROM eda_video_source_phase_quality
    UNION ALL BY NAME SELECT * FROM eda_problem_source_phase_quality
    UNION ALL BY NAME SELECT * FROM eda_comment_source_phase_quality
    UNION ALL BY NAME SELECT * FROM eda_reply_source_phase_quality
    """,
)

write_relation(
    "eda_source_time_audit",
    "source/eda_source_time_audit",
    """
    SELECT
        source_name,
        max(source_available) AS source_available,
        sum(event_count) AS total_events,
        sum(event_count) FILTER (WHERE time_bucket IN ('P1','P2','P3','P4','P5'))
          AS events_in_study_window,
        sum(event_count) FILTER (WHERE time_bucket = 'BEFORE_T0') AS events_before_t0,
        sum(event_count) FILTER (WHERE time_bucket = 'AFTER_T5') AS events_after_t5,
        sum(invalid_time_count) AS invalid_time_count,
        min(min_event_time) AS min_event_time,
        max(max_event_time) AS max_event_time,
        count(DISTINCT time_bucket) FILTER (
          WHERE time_bucket IN ('P1','P2','P3','P4','P5') AND event_count > 0
        ) AS n_phases_with_events
    FROM eda_source_phase_coverage
    GROUP BY source_name
    """,
)



## 5. Coverage resource/problem/comment → course/chapter



In [ ]:
# Mỗi source được build/checkpoint tuần tự. Video/problem được gom theo
# resource trước khi join để tránh join hàng triệu event với mapping catalog.
write_relation(
    "eda_mapping_coverage_video",
    "quality/mapping_coverage_parts/video",
    f"""
    WITH resource_frequency AS (
      SELECT
          {video_id} AS resource_id,
          CAST(sum(COALESCE(array_length(video_item.segment), 0)) AS BIGINT)
            AS event_count
      FROM (
        SELECT * FROM src_user_video
        USING SAMPLE {VIDEO_COVERAGE_SAMPLE_PERCENT} PERCENT (bernoulli, 2020)
      ) uv
      CROSS JOIN UNNEST(uv.seq) AS s(video_item)
      GROUP BY 1
    )
    SELECT
        'video' AS source_name,
        'segments' AS unit_type,
        'sampled_user_rows' AS coverage_mode,
        {VIDEO_COVERAGE_SAMPLE_PERCENT}::DOUBLE AS sample_percent,
        CAST(COALESCE(sum(f.event_count), 0) AS BIGINT) AS total_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.n_course_matches = 1 AND m.n_lo_matches = 1
        ), 0) AS BIGINT) AS strict_unique_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.video_id IS NULL
        ), 0) AS BIGINT) AS unmapped_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.video_id IS NOT NULL
            AND (m.n_course_matches <> 1 OR m.n_lo_matches <> 1)
        ), 0) AS BIGINT) AS ambiguous_units
    FROM resource_frequency f
    LEFT JOIN map_video_lo m ON f.resource_id = m.video_id
    """,
)

write_relation(
    "eda_mapping_coverage_problem",
    "quality/mapping_coverage_parts/problem",
    f"""
    WITH resource_frequency AS (
      SELECT {problem_id} AS resource_id, count(*)::BIGINT AS event_count
      FROM src_user_problem up
      GROUP BY 1
    )
    SELECT
        'problem' AS source_name,
        'submission_logs' AS unit_type,
        'exact' AS coverage_mode,
        100.0::DOUBLE AS sample_percent,
        CAST(COALESCE(sum(f.event_count), 0) AS BIGINT) AS total_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.n_course_matches = 1 AND m.n_lo_matches = 1
        ), 0) AS BIGINT) AS strict_unique_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.problem_id IS NULL
        ), 0) AS BIGINT) AS unmapped_units,
        CAST(COALESCE(sum(f.event_count) FILTER (
          WHERE m.problem_id IS NOT NULL
            AND (m.n_course_matches <> 1 OR m.n_lo_matches <> 1)
        ), 0) AS BIGINT) AS ambiguous_units
    FROM resource_frequency f
    LEFT JOIN map_problem_lo m ON f.resource_id = m.problem_id
    """,
)

if AVAILABLE_RAW.get("src_comment") and AVAILABLE_MODEL.get("bridge_comment_resource"):
    comment_mapping_query = f"""
    WITH bridge_status AS (
      SELECT
          comment_id,
          CASE
            WHEN bool_or(assignment_status = 'resolved_unique_lo') THEN 'strict'
            WHEN bool_or(assignment_status = 'ambiguous_resource') THEN 'ambiguous'
            ELSE 'unmapped'
          END AS mapping_class
      FROM bridge_comment_resource
      GROUP BY comment_id
    )
    SELECT
        'comment' AS source_name,
        'comments' AS unit_type,
        'exact' AS coverage_mode,
        100.0::DOUBLE AS sample_percent,
        count(*)::BIGINT AS total_units,
        count(*) FILTER (WHERE b.mapping_class = 'strict')::BIGINT
          AS strict_unique_units,
        count(*) FILTER (WHERE b.comment_id IS NULL OR b.mapping_class = 'unmapped')::BIGINT
          AS unmapped_units,
        count(*) FILTER (WHERE b.mapping_class = 'ambiguous')::BIGINT
          AS ambiguous_units
    FROM src_comment cm
    LEFT JOIN bridge_status b ON {comment_id_expr} = b.comment_id
    """
else:
    comment_mapping_query = """
    SELECT 'comment' AS source_name, 'comments' AS unit_type,
           'source_missing' AS coverage_mode, 0.0::DOUBLE AS sample_percent,
           0::BIGINT AS total_units, 0::BIGINT AS strict_unique_units,
           0::BIGINT AS unmapped_units, 0::BIGINT AS ambiguous_units
    """

write_relation(
    "eda_mapping_coverage_comment",
    "quality/mapping_coverage_parts/comment",
    comment_mapping_query,
)

write_relation(
    "eda_mapping_coverage",
    "quality/eda_mapping_coverage",
    """
    WITH combined AS (
      SELECT * FROM eda_mapping_coverage_video
      UNION ALL BY NAME SELECT * FROM eda_mapping_coverage_problem
      UNION ALL BY NAME SELECT * FROM eda_mapping_coverage_comment
    )
    SELECT *, strict_unique_units::DOUBLE / nullif(total_units, 0)
      AS strict_unique_rate
    FROM combined
    """,
)



## 6. Activity user × course × chapter × phase theo từng source

Các bảng này là activity facts phục vụ EDA, chưa phải feature/indicator facts.
Video/problem dùng mapping strict. Comment chỉ xuống chapter khi resource resolve
duy nhất; comment còn lại vẫn được giữ ở bảng user–course.

Video và problem được chia theo `hash(user_id)`. Mỗi shard chỉ giữ khoảng
`1 / USER_HASH_SHARDS` số user trong hash table GROUP BY và được checkpoint riêng.
Trước khi aggregate, notebook đọc mỗi source gốc đúng một lần và partition nó
theo `user_shard` trong `/kaggle/temp`. Nhờ vậy 256 shard không đồng nghĩa với
việc quét lại toàn bộ file raw 256 lần. Video được lọc mapping strict trước
khi bung danh sách segment để giảm peak RAM.
Ở low-memory mode, notebook không tính distinct video/problem/active-day trong
cùng phép GROUP BY lớn; các field đó để `null`. Chúng không cần để quyết định
activity/eligibility và sẽ được tính ở notebook feature sau nếu thực sự cần.



In [ ]:
SOURCE_SHARD_ROOT = scratch_base / "moocubex_eda_source_shards"


def ensure_source_user_shards(
    output_name: str,
    source_view: str,
    source_alias: str,
    normalized_user_expr: str,
    shard_count: int = USER_HASH_SHARDS,
) -> Path:
    """Partition raw rows một lần để mỗi aggregate shard không quét lại toàn source."""
    out_dir = SOURCE_SHARD_ROOT / output_name
    metadata_path = out_dir / "_SHARD_LAYOUT.json"
    layout_ok = False
    if metadata_path.is_file() and any(out_dir.rglob("*.parquet")):
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            layout_ok = int(metadata.get("shard_count", -1)) == shard_count
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            layout_ok = False

    if not layout_ok:
        if out_dir.exists():
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"STAGE {output_name}: partition raw source thành {shard_count} user shards")
        started = time.time()
        con.execute(
            f"""
            COPY (
              SELECT
                  mod(hash({normalized_user_expr}), {shard_count})::INTEGER AS user_shard,
                  {source_alias}.*
              FROM {source_view} {source_alias}
              WHERE {normalized_user_expr} IS NOT NULL
            )
            TO {sql_string(str(out_dir))}
            (FORMAT PARQUET, PARTITION_BY (user_shard), COMPRESSION ZSTD,
             COMPRESSION_LEVEL {ZSTD_LEVEL}, ROW_GROUP_SIZE 10000)
            """
        )
        metadata_path.write_text(
            json.dumps({"shard_count": shard_count, "source_view": source_view}),
            encoding="utf-8",
        )
        print(f" OK staging {output_name}: {(time.time() - started) / 60:.1f} phút")
    else:
        print(f"SKIP staging {output_name}: layout {shard_count} shards đã tồn tại")

    glob_path = str(out_dir / "**" / "*.parquet")
    con.execute(
        f"CREATE OR REPLACE VIEW {output_name} AS "
        f"SELECT * FROM read_parquet({sql_string(glob_path)}, "
        f"union_by_name=true, hive_partitioning=true)"
    )
    return out_dir


ensure_source_user_shards(
    "src_user_video_sharded",
    "src_user_video",
    "uv0",
    normalize_id("uv0.user_id", "U_"),
)
ensure_source_user_shards(
    "src_user_problem_sharded",
    "src_user_problem",
    "up0",
    normalize_id("up0.user_id", "U_"),
)


def build_video_activity_query(user_shard: int, shard_count: int) -> str:
    return f"""
    WITH source_users AS (
      SELECT *
      FROM src_user_video_sharded uv0
      WHERE uv0.user_shard = {user_shard}
    ), strict_video_items AS (
      SELECT
          {video_user_id} AS user_id,
          m.course_id_candidate AS course_id,
          m.lo_key_candidate AS lo_key,
          m.chapter_major_candidate AS chapter_major,
          video_item.segment AS segments
      FROM source_users uv
      CROSS JOIN UNNEST(uv.seq) AS s(video_item)
      JOIN map_video_lo m ON {video_id} = m.video_id
      WHERE m.n_course_matches = 1 AND m.n_lo_matches = 1
    ), flat AS (
      SELECT
          v.user_id, v.course_id, v.lo_key, v.chapter_major,
          segment_item.start_point AS start_point,
          segment_item.end_point AS end_point,
          segment_item.speed AS speed,
          CASE
            WHEN segment_item.local_start_time BETWEEN 0 AND 4102444800
              THEN CAST(to_timestamp(segment_item.local_start_time) AS TIMESTAMP)
            ELSE NULL
          END AS event_time
      FROM strict_video_items v
      CROSS JOIN UNNEST(v.segments) AS g(segment_item)
    )
    SELECT
        {user_shard}::INTEGER AS user_shard,
        a.user_id, a.course_id, a.lo_key, a.chapter_major,
        p.phase_id, p.phase_num, p.phase_start, p.phase_end,
        count(*) AS n_video_segments,
        CAST(NULL AS BIGINT) AS n_videos_touched,
        CAST(NULL AS BIGINT) AS n_video_active_days,
        min(a.event_time) AS first_video_event_time,
        max(a.event_time) AS last_video_event_time,
        count(*) FILTER (
          WHERE a.start_point IS NULL OR a.end_point IS NULL OR a.end_point <= a.start_point
        ) AS n_invalid_video_intervals,
        count(*) FILTER (WHERE a.speed IS NULL OR a.speed <= 0) AS n_invalid_video_speeds,
        sum(a.end_point - a.start_point) FILTER (
          WHERE a.start_point IS NOT NULL AND a.end_point > a.start_point
        ) AS raw_watch_seconds,
        sum((a.end_point - a.start_point) / a.speed) FILTER (
          WHERE a.start_point IS NOT NULL AND a.end_point > a.start_point AND a.speed > 0
        ) AS playback_adjusted_watch_seconds
    FROM flat a
    JOIN phase_dim p ON a.event_time >= p.phase_start AND a.event_time < p.phase_end
    WHERE a.user_id IS NOT NULL
    GROUP BY a.user_id, a.course_id, a.lo_key, a.chapter_major,
             p.phase_id, p.phase_num, p.phase_start, p.phase_end
    """


write_hash_sharded_relation(
    "eda_user_course_chapter_phase_video",
    "activity/eda_user_course_chapter_phase_video",
    build_video_activity_query,
)


def build_problem_activity_query(user_shard: int, shard_count: int) -> str:
    return f"""
    WITH source_users AS (
      SELECT *
      FROM src_user_problem_sharded up0
      WHERE up0.user_shard = {user_shard}
    ), typed AS (
      SELECT
          {problem_user_id} AS user_id,
          {problem_id} AS problem_id,
          try_cast(up.submit_time AS TIMESTAMP) AS event_time,
          up.is_correct,
          up.attempts,
          up.score
      FROM source_users up
    ), assigned AS (
      SELECT
          t.*, m.course_id_candidate AS course_id,
          m.lo_key_candidate AS lo_key,
          m.chapter_major_candidate AS chapter_major
      FROM typed t
      JOIN map_problem_lo m USING (problem_id)
      WHERE m.n_course_matches = 1 AND m.n_lo_matches = 1
    )
    SELECT
        {user_shard}::INTEGER AS user_shard,
        a.user_id, a.course_id, a.lo_key, a.chapter_major,
        p.phase_id, p.phase_num, p.phase_start, p.phase_end,
        count(*) AS n_problem_logs,
        CAST(NULL AS BIGINT) AS n_problems_touched,
        CAST(NULL AS BIGINT) AS n_problem_active_days,
        min(a.event_time) AS first_problem_event_time,
        max(a.event_time) AS last_problem_event_time,
        sum(CASE WHEN a.is_correct = 1 THEN 1 ELSE 0 END) AS n_correct_logs,
        sum(COALESCE(a.attempts, 0)) AS attempts_sum,
        count(*) FILTER (WHERE a.score IS NULL) AS n_null_scores,
        count(*) FILTER (WHERE a.score < 0) AS n_negative_scores,
        avg(a.score) FILTER (WHERE a.score >= 0) AS mean_valid_score
    FROM assigned a
    JOIN phase_dim p ON a.event_time >= p.phase_start AND a.event_time < p.phase_end
    WHERE a.user_id IS NOT NULL
    GROUP BY a.user_id, a.course_id, a.lo_key, a.chapter_major,
             p.phase_id, p.phase_num, p.phase_start, p.phase_end
    """


write_hash_sharded_relation(
    "eda_user_course_chapter_phase_problem",
    "activity/eda_user_course_chapter_phase_problem",
    build_problem_activity_query,
)

if AVAILABLE_RAW.get("src_comment") and AVAILABLE_MODEL.get("bridge_comment_resource"):
    comment_lo_query = f"""
    WITH typed AS (
      SELECT
          {comment_id_expr} AS comment_id,
          COALESCE(
            {comment_user_id_expr},
            CASE WHEN u.n_user_matches = 1 THEN u.user_id_candidate END
          ) AS user_id,
          try_cast(cm.create_time AS TIMESTAMP) AS event_time,
          cm.text
      FROM src_comment cm
      LEFT JOIN map_comment_user u ON {comment_id_expr} = u.comment_id
    ), assigned AS (
      SELECT
          t.*, b.resolved_course_id AS course_id, b.resolved_lo_key AS lo_key
      FROM typed t
      JOIN bridge_comment_resource b USING (comment_id)
      WHERE b.assignment_status = 'resolved_unique_lo'
    )
    SELECT
        mod(hash(a.user_id), {USER_HASH_SHARDS})::INTEGER AS user_shard,
        a.user_id, a.course_id, a.lo_key, l.chapter_major,
        p.phase_id, p.phase_num, p.phase_start, p.phase_end,
        count(*) AS n_comments_raw,
        count(*) FILTER (WHERE a.text IS NOT NULL AND length(trim(a.text)) > 0)
          AS n_comments_nonempty,
        count(*) FILTER (WHERE a.text IS NULL OR length(trim(a.text)) < 5)
          AS n_comments_short,
        min(a.event_time) AS first_comment_time,
        max(a.event_time) AS last_comment_time
    FROM assigned a
    JOIN dim_lo_catalog l USING (course_id, lo_key)
    JOIN phase_dim p ON a.event_time >= p.phase_start AND a.event_time < p.phase_end
    WHERE a.user_id IS NOT NULL
    GROUP BY a.user_id, a.course_id, a.lo_key, l.chapter_major,
             p.phase_id, p.phase_num, p.phase_start, p.phase_end
    """
else:
    comment_lo_query = """
    SELECT CAST(NULL AS INTEGER) AS user_shard,
           CAST(NULL AS VARCHAR) AS user_id, CAST(NULL AS VARCHAR) AS course_id,
           CAST(NULL AS VARCHAR) AS lo_key, CAST(NULL AS BIGINT) AS chapter_major,
           CAST(NULL AS VARCHAR) AS phase_id, CAST(NULL AS INTEGER) AS phase_num,
           CAST(NULL AS TIMESTAMP) AS phase_start, CAST(NULL AS TIMESTAMP) AS phase_end,
           CAST(0 AS BIGINT) AS n_comments_raw,
           CAST(0 AS BIGINT) AS n_comments_nonempty,
           CAST(0 AS BIGINT) AS n_comments_short,
           CAST(NULL AS TIMESTAMP) AS first_comment_time,
           CAST(NULL AS TIMESTAMP) AS last_comment_time
    WHERE FALSE
    """
write_relation(
    "eda_user_course_chapter_phase_comment",
    "activity/eda_user_course_chapter_phase_comment",
    comment_lo_query,
    required_columns={"user_shard"},
)

if AVAILABLE_RAW.get("src_comment"):
    comment_course_query = f"""
    WITH typed AS (
      SELECT
          {comment_id_expr} AS comment_id,
          COALESCE(
            {comment_user_id_expr},
            CASE WHEN u.n_user_matches = 1 THEN u.user_id_candidate END
          ) AS user_id,
          try_cast(cm.create_time AS TIMESTAMP) AS event_time,
          cm.text
      FROM src_comment cm
      LEFT JOIN map_comment_user u ON {comment_id_expr} = u.comment_id
    ), assigned AS (
      SELECT t.*, m.course_id_candidate AS course_id
      FROM typed t
      JOIN map_comment_course m USING (comment_id)
      WHERE m.n_course_matches = 1
    )
    SELECT
        mod(hash(a.user_id), {USER_HASH_SHARDS})::INTEGER AS user_shard,
        a.user_id, a.course_id,
        p.phase_id, p.phase_num, p.phase_start, p.phase_end,
        count(*) AS n_comments_raw,
        count(*) FILTER (WHERE a.text IS NOT NULL AND length(trim(a.text)) > 0)
          AS n_comments_nonempty,
        count(*) FILTER (WHERE a.text IS NULL OR length(trim(a.text)) < 5)
          AS n_comments_short,
        min(a.event_time) AS first_comment_time,
        max(a.event_time) AS last_comment_time
    FROM assigned a
    JOIN phase_dim p ON a.event_time >= p.phase_start AND a.event_time < p.phase_end
    WHERE a.user_id IS NOT NULL
    GROUP BY a.user_id, a.course_id,
             p.phase_id, p.phase_num, p.phase_start, p.phase_end
    """
else:
    comment_course_query = """
    SELECT CAST(NULL AS INTEGER) AS user_shard,
           CAST(NULL AS VARCHAR) AS user_id, CAST(NULL AS VARCHAR) AS course_id,
           CAST(NULL AS VARCHAR) AS phase_id, CAST(NULL AS INTEGER) AS phase_num,
           CAST(NULL AS TIMESTAMP) AS phase_start, CAST(NULL AS TIMESTAMP) AS phase_end,
           CAST(0 AS BIGINT) AS n_comments_raw,
           CAST(0 AS BIGINT) AS n_comments_nonempty,
           CAST(0 AS BIGINT) AS n_comments_short,
           CAST(NULL AS TIMESTAMP) AS first_comment_time,
           CAST(NULL AS TIMESTAMP) AS last_comment_time
    WHERE FALSE
    """
write_relation(
    "eda_user_course_phase_comment",
    "activity/eda_user_course_phase_comment",
    comment_course_query,
    required_columns={"user_shard"},
)



## 7. Hợp nhất activity facts ở ba tầng monitor

- `has_learning_activity`: video hoặc problem.
- `has_social_activity`: comment.
- `has_any_activity`: learning hoặc social.

Quality score vẫn để trống ở bước này; inactivity không được thay bằng điểm 0.



In [ ]:
def build_combined_user_chapter_query(user_shard: int, shard_count: int) -> str:
    return f"""
    WITH stacked AS (
      SELECT
          user_id, course_id, lo_key, chapter_major,
          phase_id, phase_num, phase_start, phase_end,
          TRUE AS has_video_activity, FALSE AS has_problem_activity,
          FALSE AS has_social_activity,
          n_video_segments, n_videos_touched,
          CAST(0 AS BIGINT) AS n_problem_logs,
          CAST(0 AS BIGINT) AS n_problems_touched,
          CAST(0 AS BIGINT) AS n_comments_raw,
          first_video_event_time AS first_event_time,
          last_video_event_time AS last_event_time
      FROM eda_user_course_chapter_phase_video
      WHERE user_shard = {user_shard}

      UNION ALL BY NAME

      SELECT
          user_id, course_id, lo_key, chapter_major,
          phase_id, phase_num, phase_start, phase_end,
          FALSE AS has_video_activity, TRUE AS has_problem_activity,
          FALSE AS has_social_activity,
          CAST(0 AS BIGINT) AS n_video_segments,
          CAST(0 AS BIGINT) AS n_videos_touched,
          n_problem_logs, n_problems_touched,
          CAST(0 AS BIGINT) AS n_comments_raw,
          first_problem_event_time AS first_event_time,
          last_problem_event_time AS last_event_time
      FROM eda_user_course_chapter_phase_problem
      WHERE user_shard = {user_shard}

      UNION ALL BY NAME

      SELECT
          user_id, course_id, lo_key, chapter_major,
          phase_id, phase_num, phase_start, phase_end,
          FALSE AS has_video_activity, FALSE AS has_problem_activity,
          TRUE AS has_social_activity,
          CAST(0 AS BIGINT) AS n_video_segments,
          CAST(0 AS BIGINT) AS n_videos_touched,
          CAST(0 AS BIGINT) AS n_problem_logs,
          CAST(0 AS BIGINT) AS n_problems_touched,
          n_comments_raw,
          first_comment_time AS first_event_time,
          last_comment_time AS last_event_time
      FROM eda_user_course_chapter_phase_comment
      WHERE user_shard = {user_shard}
    )
    SELECT
        {user_shard}::INTEGER AS user_shard,
        user_id, course_id, lo_key, chapter_major,
        phase_id, phase_num, min(phase_start) AS phase_start, min(phase_end) AS phase_end,
        max(has_video_activity) AS has_video_activity,
        max(has_problem_activity) AS has_problem_activity,
        max(has_social_activity) AS has_social_activity,
        (max(has_video_activity) OR max(has_problem_activity)) AS has_learning_activity,
        (max(has_video_activity) OR max(has_problem_activity) OR max(has_social_activity))
          AS has_any_activity,
        sum(n_video_segments) AS n_video_segments,
        sum(n_videos_touched) AS n_videos_touched,
        sum(n_problem_logs) AS n_problem_logs,
        sum(n_problems_touched) AS n_problems_touched,
        sum(n_comments_raw) AS n_comments_raw,
        min(first_event_time) AS first_event_time,
        max(last_event_time) AS last_event_time
    FROM stacked
    GROUP BY user_id, course_id, lo_key, chapter_major, phase_id, phase_num
    """


write_hash_sharded_relation(
    "eda_user_course_chapter_phase_activity",
    "activity/eda_user_course_chapter_phase_activity",
    build_combined_user_chapter_query,
)


def build_user_course_query(user_shard: int, shard_count: int) -> str:
    return f"""
    WITH learning AS (
      SELECT
          user_id, course_id, phase_id, phase_num,
          min(phase_start) AS phase_start, min(phase_end) AS phase_end,
          max(has_video_activity) AS has_video_activity,
          max(has_problem_activity) AS has_problem_activity,
          max(has_learning_activity) AS has_learning_activity,
          count(*) FILTER (WHERE has_learning_activity) AS n_active_chapters_learning,
          min(chapter_major) FILTER (WHERE has_learning_activity) AS min_active_chapter,
          max(chapter_major) FILTER (WHERE has_learning_activity) AS max_active_chapter,
          sum(n_video_segments) AS n_video_segments,
          sum(n_problem_logs) AS n_problem_logs,
          min(first_event_time) FILTER (WHERE has_learning_activity) AS first_learning_event_time,
          max(last_event_time) FILTER (WHERE has_learning_activity) AS last_learning_event_time
      FROM eda_user_course_chapter_phase_activity
      WHERE user_shard = {user_shard}
      GROUP BY user_id, course_id, phase_id, phase_num
    ), social AS (
      SELECT
          user_id, course_id, phase_id, phase_num,
          TRUE AS has_social_activity,
          n_comments_raw,
          first_comment_time,
          last_comment_time
      FROM eda_user_course_phase_comment
      WHERE user_shard = {user_shard}
    ), keys AS (
      SELECT user_id, course_id, phase_id, phase_num FROM learning
      UNION
      SELECT user_id, course_id, phase_id, phase_num FROM social
    )
    SELECT
        {user_shard}::INTEGER AS user_shard,
        k.user_id, k.course_id, k.phase_id, k.phase_num,
        COALESCE(l.phase_start, p.phase_start) AS phase_start,
        COALESCE(l.phase_end, p.phase_end) AS phase_end,
        COALESCE(l.has_video_activity, FALSE) AS has_video_activity,
        COALESCE(l.has_problem_activity, FALSE) AS has_problem_activity,
        COALESCE(s.has_social_activity, FALSE) AS has_social_activity,
        COALESCE(l.has_learning_activity, FALSE) AS has_learning_activity,
        (COALESCE(l.has_learning_activity, FALSE) OR COALESCE(s.has_social_activity, FALSE))
          AS has_any_activity,
        COALESCE(l.n_active_chapters_learning, 0) AS n_active_chapters_learning,
        l.min_active_chapter, l.max_active_chapter,
        COALESCE(l.n_video_segments, 0) AS n_video_segments,
        COALESCE(l.n_problem_logs, 0) AS n_problem_logs,
        COALESCE(s.n_comments_raw, 0) AS n_comments_raw,
        least(l.first_learning_event_time, s.first_comment_time) AS first_event_time,
        greatest(l.last_learning_event_time, s.last_comment_time) AS last_event_time
    FROM keys k
    JOIN phase_dim p USING (phase_id, phase_num)
    LEFT JOIN learning l USING (user_id, course_id, phase_id, phase_num)
    LEFT JOIN social s USING (user_id, course_id, phase_id, phase_num)
    """


write_hash_sharded_relation(
    "eda_user_course_phase_activity",
    "activity/eda_user_course_phase_activity",
    build_user_course_query,
)

write_relation(
    "eda_course_chapter_phase_activity",
    "activity/eda_course_chapter_phase_activity",
    """
    WITH observed AS (
      SELECT
          course_id, lo_key, chapter_major, phase_id, phase_num,
          count(*) FILTER (WHERE has_learning_activity) AS n_learning_users,
          count(*) FILTER (WHERE has_social_activity) AS n_social_users,
          count(*) FILTER (WHERE has_any_activity) AS n_any_active_users,
          sum(n_video_segments) AS n_video_segments,
          sum(n_problem_logs) AS n_problem_logs,
          sum(n_comments_raw) AS n_comments_raw,
          min(first_event_time) AS first_event_time,
          max(last_event_time) AS last_event_time
      FROM eda_user_course_chapter_phase_activity
      GROUP BY course_id, lo_key, chapter_major, phase_id, phase_num
    ), source_availability AS (
      SELECT
          time_bucket AS phase_id,
          max(source_name = 'video' AND event_count > 0) AS is_video_source_observed_phase,
          max(source_name = 'problem' AND event_count > 0) AS is_problem_source_observed_phase,
          max(source_name = 'comment' AND event_count > 0) AS is_comment_source_observed_phase,
          max(source_name = 'reply' AND event_count > 0) AS is_reply_source_observed_phase
      FROM eda_source_phase_coverage
      WHERE time_bucket IN ('P1','P2','P3','P4','P5')
      GROUP BY time_bucket
    ), skeleton AS (
      SELECT
          l.course_id, l.course_name, l.lo_key, l.chapter_major,
          l.n_videos_available, l.n_exercises_available, l.n_problems_available,
          p.phase_id, p.phase_num, p.phase_start, p.phase_end
      FROM dim_lo_catalog l
      CROSS JOIN phase_dim p
    )
    SELECT
        s.*,
        COALESCE(o.n_learning_users, 0) AS n_learning_users,
        COALESCE(o.n_social_users, 0) AS n_social_users,
        COALESCE(o.n_any_active_users, 0) AS n_any_active_users,
        COALESCE(o.n_video_segments, 0) AS n_video_segments,
        COALESCE(o.n_problem_logs, 0) AS n_problem_logs,
        COALESCE(o.n_comments_raw, 0) AS n_comments_raw,
        o.first_event_time, o.last_event_time,
        COALESCE(sa.is_video_source_observed_phase, FALSE) AS is_video_source_observed_phase,
        COALESCE(sa.is_problem_source_observed_phase, FALSE) AS is_problem_source_observed_phase,
        COALESCE(sa.is_comment_source_observed_phase, FALSE) AS is_comment_source_observed_phase,
        COALESCE(sa.is_reply_source_observed_phase, FALSE) AS is_reply_source_observed_phase,
        NOT (
          COALESCE(sa.is_video_source_observed_phase, FALSE)
          OR COALESCE(sa.is_problem_source_observed_phase, FALSE)
        ) AS is_learning_source_censored_phase,
        COALESCE(o.n_learning_users, 0) > 0 AS has_learning_activity,
        COALESCE(o.n_social_users, 0) > 0 AS has_social_activity,
        COALESCE(o.n_any_active_users, 0) > 0 AS has_any_activity,
        CASE
          WHEN COALESCE(o.n_learning_users, 0) > 0 THEN 'ACTIVE_LEARNING'
          WHEN COALESCE(o.n_social_users, 0) > 0 THEN 'SOCIAL_ONLY'
          ELSE 'NO_OBSERVED_ACTIVITY'
        END AS chapter_phase_status
    FROM skeleton s
    LEFT JOIN observed o USING (course_id, lo_key, chapter_major, phase_id, phase_num)
    LEFT JOIN source_availability sa USING (phase_id)
    """,
)

write_relation(
    "eda_course_phase_activity",
    "activity/eda_course_phase_activity",
    """
    WITH catalog AS (
      SELECT course_id, min(course_name) AS course_name,
             count(DISTINCT lo_key) AS n_catalog_chapters
      FROM dim_lo_catalog
      GROUP BY course_id
    ), chapter AS (
      SELECT
          course_id, phase_id, phase_num,
          count(*) FILTER (WHERE has_learning_activity) AS n_active_chapters_learning,
          count(*) FILTER (WHERE has_social_activity) AS n_active_chapters_social,
          sum(n_video_segments) AS n_video_segments,
          sum(n_problem_logs) AS n_problem_logs,
          max(is_video_source_observed_phase) AS is_video_source_observed_phase,
          max(is_problem_source_observed_phase) AS is_problem_source_observed_phase,
          max(is_comment_source_observed_phase) AS is_comment_source_observed_phase,
          max(is_reply_source_observed_phase) AS is_reply_source_observed_phase,
          min(first_event_time) AS first_chapter_event_time,
          max(last_event_time) AS last_chapter_event_time
      FROM eda_course_chapter_phase_activity
      GROUP BY course_id, phase_id, phase_num
    ), users AS (
      SELECT
          course_id, phase_id, phase_num,
          count(*) FILTER (WHERE has_learning_activity) AS n_learning_users,
          count(*) FILTER (WHERE has_social_activity) AS n_social_users,
          count(*) FILTER (WHERE has_any_activity) AS n_any_active_users,
          min(first_event_time) AS first_user_event_time,
          max(last_event_time) AS last_user_event_time
      FROM eda_user_course_phase_activity
      GROUP BY course_id, phase_id, phase_num
    ), enrollment AS (
      SELECT
          c.course_id, p.phase_id, p.phase_num,
          approx_count_distinct(e.user_id) FILTER (
            WHERE e.enroll_time IS NOT NULL AND e.enroll_time < p.phase_end
          ) AS n_enrolled_by_phase_end
      FROM catalog c
      CROSS JOIN phase_dim p
      LEFT JOIN bridge_user_course_enrollment e ON c.course_id = e.course_id
      GROUP BY c.course_id, p.phase_id, p.phase_num
    ), skeleton AS (
      SELECT c.*, p.phase_id, p.phase_num, p.phase_start, p.phase_end
      FROM catalog c CROSS JOIN phase_dim p
    )
    SELECT
        s.*,
        COALESCE(ch.n_active_chapters_learning, 0) AS n_active_chapters_learning,
        COALESCE(ch.n_active_chapters_social, 0) AS n_active_chapters_social,
        COALESCE(u.n_learning_users, 0) AS n_learning_users,
        COALESCE(u.n_social_users, 0) AS n_social_users,
        COALESCE(u.n_any_active_users, 0) AS n_any_active_users,
        COALESCE(e.n_enrolled_by_phase_end, 0) AS n_enrolled_by_phase_end,
        COALESCE(ch.n_video_segments, 0) AS n_video_segments,
        COALESCE(ch.n_problem_logs, 0) AS n_problem_logs,
        COALESCE(ch.is_video_source_observed_phase, FALSE) AS is_video_source_observed_phase,
        COALESCE(ch.is_problem_source_observed_phase, FALSE) AS is_problem_source_observed_phase,
        COALESCE(ch.is_comment_source_observed_phase, FALSE) AS is_comment_source_observed_phase,
        COALESCE(ch.is_reply_source_observed_phase, FALSE) AS is_reply_source_observed_phase,
        NOT (
          COALESCE(ch.is_video_source_observed_phase, FALSE)
          OR COALESCE(ch.is_problem_source_observed_phase, FALSE)
        ) AS is_learning_source_censored_phase,
        least(ch.first_chapter_event_time, u.first_user_event_time) AS first_event_time,
        greatest(ch.last_chapter_event_time, u.last_user_event_time) AS last_event_time,
        COALESCE(ch.n_active_chapters_learning, 0) > 0 AS has_learning_activity,
        COALESCE(u.n_social_users, 0) > 0 AS has_social_activity,
        CASE
          WHEN COALESCE(ch.n_active_chapters_learning, 0) > 0 THEN 'ACTIVE_LEARNING'
          WHEN COALESCE(u.n_social_users, 0) > 0 THEN 'SOCIAL_ONLY'
          ELSE 'NO_OBSERVED_ACTIVITY'
        END AS course_phase_status
    FROM skeleton s
    LEFT JOIN chapter ch USING (course_id, phase_id, phase_num)
    LEFT JOIN users u USING (course_id, phase_id, phase_num)
    LEFT JOIN enrollment e USING (course_id, phase_id, phase_num)
    """,
)



## 8. Lifecycle, inactivity, activation transition và chapter progression

`activity_vector` có thứ tự P1→P5. Một khoảng `1-0-1` là intermittent/return,
không được tự động coi phase giữa là course/chapter đã kết thúc.



In [ ]:
write_relation(
    "eda_course_chapter_lifecycle",
    "lifecycle/eda_course_chapter_lifecycle",
    """
    WITH base AS (
      SELECT
          course_id, course_name, lo_key, chapter_major,
          string_agg(CASE WHEN has_learning_activity THEN '1' ELSE '0' END,
                     '' ORDER BY phase_num) AS activity_vector,
          sum(CASE WHEN has_learning_activity THEN 1 ELSE 0 END) AS n_active_phases,
          min(phase_num) FILTER (WHERE has_learning_activity) AS first_active_phase_num,
          max(phase_num) FILTER (WHERE has_learning_activity) AS last_active_phase_num,
          min(first_event_time) AS first_observed_event_time,
          max(last_event_time) AS last_observed_event_time,
          max(n_learning_users) AS max_learning_users_in_phase,
          sum(n_video_segments) AS total_video_segments,
          sum(n_problem_logs) AS total_problem_logs,
          sum(n_comments_raw) AS total_comments_mapped_lo
      FROM eda_course_chapter_phase_activity
      GROUP BY course_id, course_name, lo_key, chapter_major
    ), gaps AS (
      SELECT
          b.*,
          CASE
            WHEN first_active_phase_num IS NULL THEN 0
            ELSE (last_active_phase_num - first_active_phase_num + 1) - n_active_phases
          END AS n_inactive_gaps_between_first_last
      FROM base b
    )
    SELECT
        *,
        CASE
          WHEN n_active_phases = 0 THEN 'NEVER_ACTIVE'
          WHEN n_active_phases = 5 THEN 'ACTIVE_ALL_PHASES'
          WHEN n_active_phases = 1 THEN 'SINGLE_ACTIVE_PHASE'
          WHEN n_inactive_gaps_between_first_last > 0 THEN 'INTERMITTENT_RETURN'
          WHEN first_active_phase_num > 1 AND last_active_phase_num = 5
            THEN 'STARTED_LATE_ACTIVE_TO_END'
          WHEN first_active_phase_num = 1 AND last_active_phase_num < 5
            THEN 'ENDED_EARLY_CONTIGUOUS'
          ELSE 'OTHER_CONTIGUOUS'
        END AS lifecycle_class
    FROM gaps
    """,
)

write_relation(
    "eda_course_lifecycle",
    "lifecycle/eda_course_lifecycle",
    """
    WITH base AS (
      SELECT
          course_id, course_name,
          string_agg(CASE WHEN has_learning_activity THEN '1' ELSE '0' END,
                     '' ORDER BY phase_num) AS activity_vector,
          sum(CASE WHEN has_learning_activity THEN 1 ELSE 0 END) AS n_active_phases,
          min(phase_num) FILTER (WHERE has_learning_activity) AS first_active_phase_num,
          max(phase_num) FILTER (WHERE has_learning_activity) AS last_active_phase_num,
          min(first_event_time) AS first_observed_event_time,
          max(last_event_time) AS last_observed_event_time,
          max(n_learning_users) AS max_learning_users_in_phase,
          max(n_active_chapters_learning) AS max_active_chapters_in_phase
      FROM eda_course_phase_activity
      GROUP BY course_id, course_name
    ), gaps AS (
      SELECT
          b.*,
          CASE
            WHEN first_active_phase_num IS NULL THEN 0
            ELSE (last_active_phase_num - first_active_phase_num + 1) - n_active_phases
          END AS n_inactive_gaps_between_first_last
      FROM base b
    )
    SELECT
        *,
        CASE
          WHEN n_active_phases = 0 THEN 'NEVER_ACTIVE'
          WHEN n_active_phases = 5 THEN 'ACTIVE_ALL_PHASES'
          WHEN n_active_phases = 1 THEN 'SINGLE_ACTIVE_PHASE'
          WHEN n_inactive_gaps_between_first_last > 0 THEN 'INTERMITTENT_RETURN'
          WHEN first_active_phase_num > 1 AND last_active_phase_num = 5
            THEN 'STARTED_LATE_ACTIVE_TO_END'
          WHEN first_active_phase_num = 1 AND last_active_phase_num < 5
            THEN 'ENDED_EARLY_CONTIGUOUS'
          ELSE 'OTHER_CONTIGUOUS'
        END AS lifecycle_class
    FROM gaps
    """,
)

write_relation(
    "eda_chapter_activation_transition",
    "lifecycle/eda_chapter_activation_transition",
    """
    SELECT
        a.phase_id AS from_phase_id,
        b.phase_id AS to_phase_id,
        a.phase_num AS from_phase_num,
        CASE WHEN a.has_learning_activity THEN 'ACTIVE' ELSE 'INACTIVE' END AS from_state,
        CASE WHEN b.has_learning_activity THEN 'ACTIVE' ELSE 'INACTIVE' END AS to_state,
        count(*) AS n_course_chapters
    FROM eda_course_chapter_phase_activity a
    JOIN eda_course_chapter_phase_activity b
      ON a.course_id = b.course_id
     AND a.lo_key = b.lo_key
     AND b.phase_num = a.phase_num + 1
    GROUP BY a.phase_id, b.phase_id, a.phase_num, from_state, to_state
    """,
)

def build_user_course_lifecycle_query(user_shard: int, shard_count: int) -> str:
    return f"""
    SELECT
        {user_shard}::INTEGER AS user_shard,
        user_id, course_id,
        min(phase_num) FILTER (WHERE has_learning_activity) AS first_learning_phase_num,
        max(phase_num) FILTER (WHERE has_learning_activity) AS last_learning_phase_num,
        min(phase_num) FILTER (WHERE has_social_activity) AS first_social_phase_num,
        max(phase_num) FILTER (WHERE has_social_activity) AS last_social_phase_num,
        count(*) FILTER (WHERE has_learning_activity) AS n_learning_active_phases,
        count(*) FILTER (WHERE has_social_activity) AS n_social_active_phases
    FROM eda_user_course_phase_activity
    WHERE user_shard = {user_shard}
    GROUP BY user_id, course_id
    """


write_hash_sharded_relation(
    "eda_user_course_lifecycle",
    "lifecycle/eda_user_course_lifecycle",
    build_user_course_lifecycle_query,
)


def build_user_progression_query(user_shard: int, shard_count: int) -> str:
    return f"""
    WITH active AS (
      SELECT
          user_id, course_id, phase_id, phase_num,
          min(chapter_major) FILTER (WHERE has_learning_activity) AS min_active_chapter,
          max(chapter_major) FILTER (WHERE has_learning_activity) AS max_active_chapter,
          count(*) FILTER (WHERE has_learning_activity)
            AS n_distinct_active_chapters
      FROM eda_user_course_chapter_phase_activity
      WHERE user_shard = {user_shard}
      GROUP BY user_id, course_id, phase_id, phase_num
      HAVING max(has_learning_activity)
    ), lagged AS (
      SELECT
          *,
          lag(phase_num) OVER (
            PARTITION BY user_id, course_id ORDER BY phase_num
          ) AS previous_active_phase_num,
          lag(min_active_chapter) OVER (
            PARTITION BY user_id, course_id ORDER BY phase_num
          ) AS previous_min_active_chapter,
          lag(max_active_chapter) OVER (
            PARTITION BY user_id, course_id ORDER BY phase_num
          ) AS previous_max_active_chapter
      FROM active
    )
    SELECT
        {user_shard}::INTEGER AS user_shard,
        *,
        n_distinct_active_chapters > 1 AS has_multiple_chapters_same_phase,
        CASE
          WHEN previous_max_active_chapter IS NULL THEN NULL
          ELSE max_active_chapter > previous_max_active_chapter
        END AS progressed_to_higher_chapter,
        CASE
          WHEN previous_max_active_chapter IS NULL THEN NULL
          ELSE min_active_chapter < previous_max_active_chapter
        END AS revisited_lower_chapter,
        CASE
          WHEN previous_max_active_chapter IS NULL THEN NULL
          ELSE max_active_chapter > previous_max_active_chapter + 1
        END AS jumped_over_at_least_one_major_chapter,
        phase_num - previous_active_phase_num AS phases_since_previous_activity
    FROM lagged
    """


write_hash_sharded_relation(
    "eda_user_course_phase_progression",
    "lifecycle/eda_user_course_phase_progression",
    build_user_progression_query,
)

write_relation(
    "eda_user_progression_summary",
    "lifecycle/eda_user_progression_summary",
    """
    SELECT
        phase_id, phase_num,
        count(*) AS n_active_user_courses,
        count(*) FILTER (WHERE has_multiple_chapters_same_phase) AS n_multi_chapter,
        count(*) FILTER (WHERE progressed_to_higher_chapter) AS n_progressed,
        count(*) FILTER (WHERE revisited_lower_chapter) AS n_revisited,
        count(*) FILTER (WHERE jumped_over_at_least_one_major_chapter) AS n_jumped,
        avg(has_multiple_chapters_same_phase::INTEGER) AS multi_chapter_rate,
        avg(progressed_to_higher_chapter::INTEGER) AS progression_rate_among_comparable,
        avg(revisited_lower_chapter::INTEGER) AS revisit_rate_among_comparable,
        avg(jumped_over_at_least_one_major_chapter::INTEGER) AS jump_rate_among_comparable
    FROM eda_user_course_phase_progression
    GROUP BY phase_id, phase_num
    ORDER BY phase_num
    """,
)



## 9. Support distribution và candidate eligibility tại mỗi cutoff

Bảng eligibility giữ nhiều rule song song; notebook không tự chọn rule cuối.
`eligible_any_prior_learning` là skeleton rộng nhất cho chiến lược dự đoán toàn
bộ chapter của một course đã từng hoạt động.



In [ ]:
write_relation(
    "eda_support_quantiles",
    "eligibility/eda_support_quantiles",
    """
    SELECT
        phase_id, phase_num,
        count(*) FILTER (WHERE has_learning_activity) AS n_active_course_chapters,
        min(n_learning_users) FILTER (WHERE has_learning_activity) AS min_users,
        quantile_cont(n_learning_users, 0.10) FILTER (WHERE has_learning_activity) AS p10_users,
        quantile_cont(n_learning_users, 0.25) FILTER (WHERE has_learning_activity) AS p25_users,
        quantile_cont(n_learning_users, 0.50) FILTER (WHERE has_learning_activity) AS median_users,
        quantile_cont(n_learning_users, 0.75) FILTER (WHERE has_learning_activity) AS p75_users,
        quantile_cont(n_learning_users, 0.90) FILTER (WHERE has_learning_activity) AS p90_users,
        max(n_learning_users) FILTER (WHERE has_learning_activity) AS max_users
    FROM eda_course_chapter_phase_activity
    GROUP BY phase_id, phase_num
    ORDER BY phase_num
    """,
)

write_relation(
    "eda_course_cutoff_eligibility_features",
    "eligibility/eda_course_cutoff_eligibility_features",
    """
    WITH cutoff AS (
      SELECT phase_id AS cutoff_phase_id, phase_num AS cutoff_phase_num,
             phase_end AS cutoff_time
      FROM phase_dim
      WHERE phase_num <= 4
    ), catalog AS (
      SELECT course_id, min(course_name) AS course_name,
             count(DISTINCT lo_key) AS n_catalog_chapters
      FROM dim_lo_catalog
      GROUP BY course_id
    ), user_cumulative AS (
      SELECT
          c.cutoff_phase_id, c.cutoff_phase_num, a.course_id,
          count(*) FILTER (WHERE a.first_learning_phase_num <= c.cutoff_phase_num)
            AS n_cumulative_learning_users,
          count(*) FILTER (WHERE a.first_social_phase_num <= c.cutoff_phase_num)
            AS n_cumulative_social_users
      FROM cutoff c
      JOIN eda_user_course_lifecycle a
        ON a.first_learning_phase_num <= c.cutoff_phase_num
        OR a.first_social_phase_num <= c.cutoff_phase_num
      GROUP BY c.cutoff_phase_id, c.cutoff_phase_num, a.course_id
    ), chapter_cumulative AS (
      SELECT
          c.cutoff_phase_id, c.cutoff_phase_num, a.course_id,
          count(DISTINCT a.lo_key) FILTER (WHERE a.has_learning_activity)
            AS n_observed_chapters_cumulative,
          sum(a.n_video_segments) AS cumulative_video_segments,
          sum(a.n_problem_logs) AS cumulative_problem_logs
      FROM cutoff c
      JOIN eda_course_chapter_phase_activity a ON a.phase_num <= c.cutoff_phase_num
      GROUP BY c.cutoff_phase_id, c.cutoff_phase_num, a.course_id
    ), recent AS (
      SELECT
          phase_id AS cutoff_phase_id, phase_num AS cutoff_phase_num, course_id,
          n_learning_users AS n_recent_learning_users,
          n_active_chapters_learning AS n_recent_active_chapters,
          has_learning_activity AS has_recent_learning_activity
      FROM eda_course_phase_activity
      WHERE phase_num <= 4
    ), skeleton AS (
      SELECT c.*, k.cutoff_phase_id, k.cutoff_phase_num, k.cutoff_time
      FROM catalog c CROSS JOIN cutoff k
    )
    SELECT
        s.*,
        COALESCE(u.n_cumulative_learning_users, 0) AS n_cumulative_learning_users,
        COALESCE(u.n_cumulative_social_users, 0) AS n_cumulative_social_users,
        COALESCE(ch.n_observed_chapters_cumulative, 0) AS n_observed_chapters_cumulative,
        COALESCE(ch.cumulative_video_segments, 0) AS cumulative_video_segments,
        COALESCE(ch.cumulative_problem_logs, 0) AS cumulative_problem_logs,
        COALESCE(r.n_recent_learning_users, 0) AS n_recent_learning_users,
        COALESCE(r.n_recent_active_chapters, 0) AS n_recent_active_chapters,
        COALESCE(r.has_recent_learning_activity, FALSE) AS has_recent_learning_activity,
        COALESCE(u.n_cumulative_learning_users, 0) > 0 AS eligible_any_prior_learning,
        COALESCE(r.n_recent_learning_users, 0) > 0 AS eligible_recent_learning,
        COALESCE(r.n_recent_learning_users, 0) >= 5 AS eligible_recent_min_5_users,
        COALESCE(u.n_cumulative_learning_users, 0) >= 5 AS eligible_cumulative_min_5_users,
        s.n_catalog_chapters - COALESCE(ch.n_observed_chapters_cumulative, 0)
          AS n_cold_unseen_chapters
    FROM skeleton s
    LEFT JOIN user_cumulative u USING (course_id, cutoff_phase_id, cutoff_phase_num)
    LEFT JOIN chapter_cumulative ch USING (course_id, cutoff_phase_id, cutoff_phase_num)
    LEFT JOIN recent r USING (course_id, cutoff_phase_id, cutoff_phase_num)
    """,
)

eligibility_rules = [
    ("ANY_PRIOR_LEARNING", "eligible_any_prior_learning"),
    ("RECENT_LEARNING", "eligible_recent_learning"),
    ("RECENT_MIN_5_USERS", "eligible_recent_min_5_users"),
    ("CUMULATIVE_MIN_5_USERS", "eligible_cumulative_min_5_users"),
]
eligibility_union = " UNION ALL ".join(
    f"""
    SELECT cutoff_phase_id, cutoff_phase_num, {sql_string(rule_name)} AS eligibility_rule,
           count(*) FILTER (WHERE {column_name}) AS n_eligible_courses,
           sum(n_catalog_chapters) FILTER (WHERE {column_name}) AS predicted_chapter_rows,
           sum(n_cold_unseen_chapters) FILTER (WHERE {column_name}) AS cold_unseen_chapter_rows
    FROM eda_course_cutoff_eligibility_features
    GROUP BY cutoff_phase_id, cutoff_phase_num
    """
    for rule_name, column_name in eligibility_rules
)
write_relation(
    "eda_eligibility_candidate_summary",
    "eligibility/eda_eligibility_candidate_summary",
    eligibility_union,
)

write_relation(
    "eda_minimum_support_candidate_summary",
    "eligibility/eda_minimum_support_candidate_summary",
    " UNION ALL ".join(
        f"""
        SELECT phase_id, phase_num, {threshold} AS minimum_active_users,
               count(*) FILTER (
                 WHERE has_learning_activity AND n_learning_users >= {threshold}
               ) AS n_valid_course_chapter_labels,
               count(*) FILTER (WHERE has_learning_activity) AS n_active_course_chapters,
               count(*) AS n_catalog_course_chapters,
               count(*) FILTER (
                 WHERE has_learning_activity AND n_learning_users >= {threshold}
               )::DOUBLE / nullif(count(*) FILTER (WHERE has_learning_activity), 0)
                 AS retained_active_label_rate
        FROM eda_course_chapter_phase_activity
        GROUP BY phase_id, phase_num
        """
        for threshold in MIN_ACTIVE_USERS_CANDIDATES
    ),
)



## 10. Bảng tóm tắt và figures

Figures chỉ dùng các bảng aggregate nhỏ nên không kéo toàn bộ fact vào pandas.



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
FIGURE_ROOT = OUTPUT_ROOT / "figures"

phase_overview = con.execute("""
WITH course AS (
  SELECT phase_id, phase_num,
         count(*) FILTER (WHERE has_learning_activity) AS active_courses
  FROM eda_course_phase_activity
  GROUP BY phase_id, phase_num
), chapter AS (
  SELECT phase_id, phase_num,
         count(*) FILTER (WHERE has_learning_activity) AS active_chapters
  FROM eda_course_chapter_phase_activity
  GROUP BY phase_id, phase_num
), users AS (
  SELECT phase_id, phase_num,
         approx_count_distinct(user_id) FILTER (WHERE has_learning_activity) AS active_users
  FROM eda_user_course_phase_activity
  GROUP BY phase_id, phase_num
)
SELECT * FROM course JOIN chapter USING (phase_id, phase_num)
JOIN users USING (phase_id, phase_num)
ORDER BY phase_num
""").fetchdf()

print("\nPHASE OVERVIEW")
print(phase_overview.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, column, title in zip(
    axes,
    ["active_courses", "active_chapters", "active_users"],
    ["Active courses", "Active chapters", "Active users"],
):
    axis.bar(phase_overview["phase_id"], phase_overview[column], color="#4472C4")
    axis.set_title(title)
    axis.set_xlabel("Phase")
    axis.ticklabel_format(axis="y", style="plain")
fig.suptitle("MOOCCubeX learning activity by fixed calendar phase")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "01_phase_activity_overview.png", dpi=160, bbox_inches="tight")
plt.show()

source_phase = con.execute("""
SELECT source_name, time_bucket, event_count
FROM eda_source_phase_coverage
WHERE time_bucket IN ('P1','P2','P3','P4','P5')
ORDER BY source_name, time_bucket
""").fetchdf()
if not source_phase.empty:
    source_pivot = source_phase.pivot(
        index="time_bucket", columns="source_name", values="event_count"
    ).fillna(0).reindex(["P1", "P2", "P3", "P4", "P5"])
    ax = source_pivot.plot(kind="bar", figsize=(10, 5), logy=True)
    ax.set_title("Event coverage by source and phase (log scale)")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Event count")
    ax.legend(title="Source")
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / "02_source_phase_coverage.png", dpi=160, bbox_inches="tight")
    plt.show()

lifecycle = con.execute("""
SELECT activity_vector, count(*) AS n_course_chapters
FROM eda_course_chapter_lifecycle
GROUP BY activity_vector
ORDER BY n_course_chapters DESC, activity_vector
LIMIT 15
""").fetchdf()
if not lifecycle.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    ordered = lifecycle.sort_values("n_course_chapters")
    ax.barh(ordered["activity_vector"], ordered["n_course_chapters"], color="#70AD47")
    ax.set_title("Top chapter activity vectors (P1→P5)")
    ax.set_xlabel("Course–chapter count")
    ax.set_ylabel("Activity vector")
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / "03_chapter_activity_vectors.png", dpi=160, bbox_inches="tight")
    plt.show()

activation_order = con.execute("""
SELECT
    chapter_major,
    COALESCE(CAST(first_active_phase_num AS VARCHAR), 'NEVER') AS first_active_phase,
    count(*) AS n_course_chapters
FROM eda_course_chapter_lifecycle
WHERE chapter_major BETWEEN 1 AND 20
GROUP BY chapter_major, first_active_phase
ORDER BY chapter_major, first_active_phase
""").fetchdf()
if not activation_order.empty:
    order_pivot = activation_order.pivot(
        index="chapter_major", columns="first_active_phase", values="n_course_chapters"
    ).fillna(0)
    for column in ["1", "2", "3", "4", "5", "NEVER"]:
        if column not in order_pivot.columns:
            order_pivot[column] = 0
    order_pivot = order_pivot[["1", "2", "3", "4", "5", "NEVER"]]
    values = order_pivot.to_numpy()
    fig, ax = plt.subplots(figsize=(9, 7))
    image = ax.imshow(values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(order_pivot.columns)), order_pivot.columns)
    ax.set_yticks(range(len(order_pivot.index)), order_pivot.index)
    ax.set_xlabel("First active phase")
    ax.set_ylabel("chapter_major")
    ax.set_title("First observed learning phase by chapter order")
    fig.colorbar(image, ax=ax, label="Course–chapter count")
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / "04_chapter_major_first_active_phase.png", dpi=160, bbox_inches="tight")
    plt.show()

eligibility_plot = con.execute("""
SELECT * FROM eda_eligibility_candidate_summary
ORDER BY cutoff_phase_num, eligibility_rule
""").fetchdf()
print("\nELIGIBILITY CANDIDATES")
print(eligibility_plot.to_string(index=False))
if not eligibility_plot.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    for rule, group in eligibility_plot.groupby("eligibility_rule"):
        group = group.sort_values("cutoff_phase_num")
        ax.plot(group["cutoff_phase_id"], group["predicted_chapter_rows"], marker="o", label=rule)
    ax.set_title("Estimated course × chapter prediction rows by cutoff")
    ax.set_xlabel("Cutoff phase")
    ax.set_ylabel("Prediction rows")
    ax.legend(title="Eligibility rule", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / "05_eligibility_matrix_size.png", dpi=160, bbox_inches="tight")
    plt.show()



## 11. Invariant checks, catalog và kết luận tự động



In [ ]:
checks = {
    "phase_count": "SELECT abs(count(*) - 5) FROM phase_dim",
    "phase_boundaries_touch": """
        SELECT count(*) FROM phase_dim a JOIN phase_dim b ON b.phase_num = a.phase_num + 1
        WHERE a.phase_end <> b.phase_start
    """,
    "duplicate_user_course_chapter_phase": """
        SELECT count(*) FROM (
          SELECT user_id, course_id, lo_key, phase_id, count(*) AS n
          FROM eda_user_course_chapter_phase_activity
          GROUP BY 1,2,3,4 HAVING count(*) > 1
        )
    """,
    "duplicate_user_course_phase": """
        SELECT count(*) FROM (
          SELECT user_id, course_id, phase_id, count(*) AS n
          FROM eda_user_course_phase_activity
          GROUP BY 1,2,3 HAVING count(*) > 1
        )
    """,
    "duplicate_course_chapter_phase": """
        SELECT count(*) FROM (
          SELECT course_id, lo_key, phase_id, count(*) AS n
          FROM eda_course_chapter_phase_activity
          GROUP BY 1,2,3 HAVING count(*) > 1
        )
    """,
    "not_five_phases_per_chapter": """
        SELECT count(*) FROM (
          SELECT course_id, lo_key, count(*) AS n
          FROM eda_course_chapter_phase_activity
          GROUP BY 1,2 HAVING count(*) <> 5
        )
    """,
    "duplicate_course_phase": """
        SELECT count(*) FROM (
          SELECT course_id, phase_id, count(*) AS n
          FROM eda_course_phase_activity
          GROUP BY 1,2 HAVING count(*) > 1
        )
    """,
    "negative_activity_counts": """
        SELECT count(*) FROM eda_course_chapter_phase_activity
        WHERE n_learning_users < 0 OR n_video_segments < 0 OR n_problem_logs < 0
    """,
}

failed_checks = []
for check_name, query in checks.items():
    violations = int(con.execute(query).fetchone()[0])
    print(f"CHECK {check_name}: violations={violations:,}")
    if violations:
        failed_checks.append((check_name, violations))
if failed_checks:
    raise AssertionError(f"Invariant tests thất bại: {failed_checks}")

source_audit = con.execute("""
SELECT * FROM eda_source_time_audit ORDER BY source_name
""").fetchdf()
mapping_audit = con.execute("""
SELECT * FROM eda_mapping_coverage ORDER BY source_name
""").fetchdf()
support_audit = con.execute("""
SELECT * FROM eda_support_quantiles ORDER BY phase_num
""").fetchdf()
progression_audit = con.execute("""
SELECT * FROM eda_user_progression_summary ORDER BY phase_num
""").fetchdf()

print("\nSOURCE TIME AUDIT")
print(source_audit.to_string(index=False))
print("\nMAPPING COVERAGE")
print(mapping_audit.to_string(index=False))
print("\nCHAPTER SUPPORT QUANTILES")
print(support_audit.to_string(index=False))
print("\nUSER CHAPTER PROGRESSION")
print(progression_audit.to_string(index=False))

successful = [row for row in build_manifest.values() if row.get("status") == "success"]
output_catalog = {
    "source_raw_root": str(RAW_ROOT_PATH),
    "source_model_root": str(MODEL_ROOT_PATH),
    "output_root": str(OUTPUT_ROOT),
    "created_with_duckdb": duckdb.__version__,
    "user_hash_shards": USER_HASH_SHARDS,
    "low_memory_mode": {
        "threads": THREADS,
        "duckdb_memory_limit_gib": memory_limit_gib,
        "memory_fraction": MEMORY_FRACTION,
        "source_user_count_method": "approx_count_distinct",
        "deferred_metrics": [
            "n_videos_touched",
            "n_video_active_days",
            "n_problems_touched",
            "n_problem_active_days",
        ],
    },
    "study_window": {
        "start": STUDY_START,
        "end_exclusive": STUDY_END_EXCLUSIVE,
        "timezone_assumption": "UTC for user-video epoch seconds",
    },
    "phase_definition": [
        {"phase_id": pid, "phase_num": pnum, "start": start, "end_exclusive": end}
        for pid, pnum, start, end in PHASES
    ],
    "activity_semantics": {
        "learning": "strict-mapped video or problem event",
        "social": "comment; LO-level only when resource maps uniquely",
        "inactive_is_not_quality_zero": True,
    },
    "mapping_coverage": {
        "video_mode": "sampled_user_rows",
        "video_sample_percent": VIDEO_COVERAGE_SAMPLE_PERCENT,
        "problem_mode": "exact",
    },
    "tables": {
        row["name"]: {
            "parquet_glob": str(OUTPUT_ROOT / row["relative_path"] / "*.parquet"),
            "row_count": row.get("row_count"),
            "output_bytes": row.get("output_bytes"),
        }
        for row in successful
    },
}
with (OUTPUT_ROOT / "catalog.json").open("w", encoding="utf-8") as file:
    json.dump(output_catalog, file, ensure_ascii=False, indent=2)

save_manifest()
(OUTPUT_ROOT / "_SUCCESS").write_text(
    "MOOCCubeX phase activity and eligibility EDA completed.\n", encoding="utf-8"
)

print("\nDONE")
print("Output:", OUTPUT_ROOT)
print("Catalog:", OUTPUT_ROOT / "catalog.json")
print("Figures:", FIGURE_ROOT)



## Cách đọc kết quả trước notebook feature/label

Chỉ chuyển sang build indicator/label sau khi xem tối thiểu:

1. `eda_source_time_audit`: source nào thực sự phủ đủ P1–P5;
2. `eda_mapping_coverage`: bao nhiêu video/problem/comment map strict xuống LO;
3. `eda_course_lifecycle` và `eda_course_chapter_lifecycle`: inactivity có phải
   gián đoạn rồi quay lại hay không;
4. `eda_eligibility_candidate_summary`: mỗi rule tạo bao nhiêu course/chapter row;
5. `eda_minimum_support_candidate_summary`: giữ được bao nhiêu nhãn nếu yêu cầu
   1/5/10/20 active users;
6. `eda_user_progression_summary`: mức độ chapter jump/revisit thực tế.

Quyết định cần chốt từ EDA:

- `course_eligible_at_cutoff`;
- định nghĩa `chapter_active_next`;
- `minimum_support` cho CQV chapter;
- AFELO-proxy được dự báo ở user/course hay đủ coverage để xuống chapter;
- course/chapter inactive nhận target activation riêng, không nhận quality zero.

Cấu hình mặc định đã tối ưu lại theo log Kaggle: 1 thread, 20% RAM DuckDB và
256 user shards. Coverage video dùng mẫu 1% user-row vì phép tính exact đã mất
hơn 5 giờ mà chỉ phục vụ EDA. Các activity fact và eligibility vẫn được tính
trên toàn bộ dữ liệu, không sample.
